In [9]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

llm=ChatGroq(model="llama-3.3-70b-versatile")

In [10]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# Graph state
class State(TypedDict):
    topic: str
    characters: str
    settings: str
    premises: str
    story_intro: str

In [4]:
# Nodes
def generate_characters(state: State):
    """Generate character descriptions"""
    msg = llm.invoke(f"Create two character names and brief traits for a story about {state['topic']}")
    return {"characters": msg.content}

def generate_setting(state: State):
    """Generate a story setting"""
    msg = llm.invoke(f"Describe a vivid setting for a story about {state['topic']}")
    return {"settings": msg.content}

def generate_premise(state: State):
    """Generate a story premise"""
    msg = llm.invoke(f"Write a one-sentence plot premise for a story about {state['topic']}")
    return {"premises": msg.content}

def combine_elements(state: State):
    """Combine characters, setting, and premise into an intro"""
    msg = llm.invoke(
        f"Write a short story introduction using these elements:\n"
        f"Characters: {state['characters']}\n"
        f"Setting: {state['settings']}\n"
        f"Premise: {state['premises']}"
    )
    return {"story_intro": msg.content}

In [7]:
# Build the graph
graph = StateGraph(State)
graph.add_node("character", generate_characters)
graph.add_node("setting", generate_setting)
graph.add_node("premise", generate_premise)
graph.add_node("combine", combine_elements)

# Define edges (parallel execution from START)
graph.add_edge(START, "character")
graph.add_edge(START, "setting")
graph.add_edge(START, "premise")
graph.add_edge("character", "combine")
graph.add_edge("setting", "combine")
graph.add_edge("premise", "combine")
graph.add_edge("combine", END)


compiled_graph = graph.compile()


In [11]:
state = {"topic": "time travel"}
result = compiled_graph.invoke(state)
print(result["story_intro"])

As the last wisps of sunlight faded from the courtyard, Evelyn "Evie" Thompson stood before the ancient clock tower, its stone façade glowing with an ethereal light. The intricate carvings that danced across its surface seemed to whisper secrets to her, drawing her closer to the threshold of the unknown. With a deep breath, she pushed open the small, ornate door to one side of the courtyard, revealing a secret chamber filled with dusty tomes and mysterious artifacts that seemed to hold the keys to unlocking the mysteries of the timestream. It was here, within the clock tower's labyrinthine heart, that Evie had spent countless hours pouring over theories and conducting experiments, driven by a personal tragedy that had left an unhealing scar on her heart.

Her latest breakthrough, a revolutionary time machine capable of sending human consciousness back in time to inhabit the body of a younger self, had finally been completed. The implications were staggering – the potential to alter the